## Convolutions for Images

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

### The Cross-Correlation Operation

In [2]:
def corr2d(X, K):  #@save
    """Compute 2D cross-correlation."""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

In [10]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
print(K.shape, K.shape[0], K.shape[1])
corr2d(X, K)

torch.Size([2, 2]) 2 2


tensor([[19., 25.],
        [37., 43.]])

In [4]:
X, "", K

(tensor([[0., 1., 2.],
         [3., 4., 5.],
         [6., 7., 8.]]),
 '',
 tensor([[0., 1.],
         [2., 3.]]))

In [7]:
X_test = torch.tensor([[0.0, 1.0], [3.0, 4.0]])
K_test = torch.tensor([[0.0, 1.0], [2.0, 3.0], [2.0, 3.0]])
corr2d(X_test, K_test)

tensor([], size=(0, 1))

### Convolutional Layers

In [11]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

### Object Edge Detection in Images

In [15]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X  # 0 - black | 1 - white

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [16]:
K = torch.tensor([[1.0, -1.0]])
K

tensor([[ 1., -1.]])

In [17]:
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [18]:
X.t()

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [19]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

### Learning a Kernel

In [26]:
# Construct a two-dimensional convolutional layer with 1 output channel and a
# kernel of shape (1, 2). For the sake of simplicity, we ignore the bias here
conv2d = nn.LazyConv2d(1, kernel_size=(1, 2), bias=False)

# The two-dimensional convolutional layer uses four-dimensional input and
# output in the format of (example, channel, height, width), where the batch
# size (number of examples in the batch) and the number of channels are both 1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # Learning rate
X, "", Y

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


(tensor([[[[1., 1., 0., 0., 0., 0., 1., 1.],
           [1., 1., 0., 0., 0., 0., 1., 1.],
           [1., 1., 0., 0., 0., 0., 1., 1.],
           [1., 1., 0., 0., 0., 0., 1., 1.],
           [1., 1., 0., 0., 0., 0., 1., 1.],
           [1., 1., 0., 0., 0., 0., 1., 1.]]]]),
 '',
 tensor([[[[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
           [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
           [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
           [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
           [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
           [ 0.,  1.,  0.,  0.,  0., -1.,  0.]]]]))

In [27]:
for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # Update the kernel
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i + 1}, loss {l.sum():.3f}')
        print(Y_hat)

epoch 2, loss 8.476
tensor([[[[ 0.1935,  0.2847,  0.0000,  0.0000,  0.0000, -0.0911,  0.1935],
          [ 0.1935,  0.2847,  0.0000,  0.0000,  0.0000, -0.0911,  0.1935],
          [ 0.1935,  0.2847,  0.0000,  0.0000,  0.0000, -0.0911,  0.1935],
          [ 0.1935,  0.2847,  0.0000,  0.0000,  0.0000, -0.0911,  0.1935],
          [ 0.1935,  0.2847,  0.0000,  0.0000,  0.0000, -0.0911,  0.1935],
          [ 0.1935,  0.2847,  0.0000,  0.0000,  0.0000, -0.0911,  0.1935]]]],
       grad_fn=<ConvolutionBackward0>)
epoch 4, loss 1.558
tensor([[[[ 0.1239,  0.7293,  0.0000,  0.0000,  0.0000, -0.6054,  0.1239],
          [ 0.1239,  0.7293,  0.0000,  0.0000,  0.0000, -0.6054,  0.1239],
          [ 0.1239,  0.7293,  0.0000,  0.0000,  0.0000, -0.6054,  0.1239],
          [ 0.1239,  0.7293,  0.0000,  0.0000,  0.0000, -0.6054,  0.1239],
          [ 0.1239,  0.7293,  0.0000,  0.0000,  0.0000, -0.6054,  0.1239],
          [ 0.1239,  0.7293,  0.0000,  0.0000,  0.0000, -0.6054,  0.1239]]]],
       grad_fn=

In [35]:
conv2d.weight.data[:], conv2d.weight.data.reshape((1, 2))

(tensor([[[[ 0.9724, -0.9984]]]]), tensor([[ 0.9724, -0.9984]]))

## Exercises

In [61]:
X = torch.zeros((6, 8))
for i in range(4):
    X[i+2:,i] = 1
    X[:4-i,-i-1] = 1
X

tensor([[0., 0., 0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 0., 0., 1., 1., 1.],
        [1., 0., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 0., 1.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.]])

In [65]:
corr2d(X, K)

tensor([[ 0.,  0.,  0., -1.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0., -1.,  0.,  0.],
        [ 1.,  0.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0.,  0., -1.],
        [ 0.,  0.,  1.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  1.,  0.,  0.,  0.]])

In [66]:
corr2d(X.t(), K)

tensor([[ 0., -1.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  0.],
        [ 0.,  0.,  0.,  0., -1.],
        [ 1.,  0.,  0.,  0.,  0.],
        [ 0.,  1.,  0.,  0.,  0.],
        [ 0.,  0.,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  1.,  0.]])

In [67]:
corr2d(X, K.t())

tensor([[ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.],
        [-1.,  0.,  0.,  0.,  0.,  1.,  0.,  0.],
        [ 0., -1.,  0.,  0.,  0.,  0.,  1.,  0.],
        [ 0.,  0., -1.,  0.,  0.,  0.,  0.,  1.],
        [ 0.,  0.,  0., -1.,  0.,  0.,  0.,  0.]])